# 1. What you'll learn

        - implement tabular Q-learning with exploration
- train CPU PPO on CartPole under a tiny budget
- show failure without exploration and compare RL with planning

        **The one question this notebook answers:** How do value updates learn a grid policy, and what changes when PPO learns a neural policy on CartPole?

# 2. Setup

This lesson keeps reinforcement learning: q-learning and ppo small enough to inspect end to end. Fixed seeds and one shared split make every comparison reproducible.

The seed is fixed at 0 for data, splits, optimization, and sampling.
Versions are printed so this execution remains an auditable experiment,
not a result detached from the software that produced it.

In [ ]:
from pathlib import Path
import platform
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
import gymnasium as gym
from stable_baselines3 import PPO
from stable_baselines3.common.callbacks import BaseCallback

SEED = 0
random.seed(SEED)
np.random.seed(SEED)
sns.set_theme(style="whitegrid", palette="colorblind")

print(f"Python: {platform.python_version()}")
print(f"NumPy: {np.__version__} | pandas: {pd.__version__} | scikit-learn: {sklearn.__version__}")
print(f"Seed: {SEED}")

# 3. The data

A deterministic 5×5 gridworld gives -0.02 per step, -1 for a trap, and +1 at the goal. CartPole-v1 supplies a continuous-state control task through Gymnasium.

**Small example:** If reward=1, next best Q=.6, discount=.9, and current Q=.2, target=1+.9×.6=1.54; with learning rate .5 the update moves Q halfway to .87.

In [ ]:
size=5;start=(0,0);goal=(4,4);trap=(2,2);actions=[(-1,0),(1,0),(0,-1),(0,1)]
def step_grid(state,a):
 r,c=state;dr,dc=actions[a];nxt=(min(max(r+dr,0),size-1),min(max(c+dc,0),size-1))
 if nxt==goal:return nxt,1.0,True
 if nxt==trap:return nxt,-1.0,True
 return nxt,-.02,False
print("Grid states/actions:",size*size,len(actions),"start/goal/trap:",start,goal,trap);env=gym.make("CartPole-v1");obs,_=env.reset(seed=SEED);print("CartPole observation:",obs,"actions:",env.action_space.n);env.close()

# 4. Explore

The three views establish the target, the useful structure, and the specific data property that will govern the model's success or failure.

The next code produces three purposeful panels. Read the written
takeaways underneath before moving on; the plots are evidence for later
modeling choices, not decoration.

In [ ]:
fig,axes=plt.subplots(1,3,figsize=(14,3.7));grid=np.zeros((size,size));grid[goal]=1;grid[trap]=-1;axes[0].imshow(grid,cmap="RdYlGn",vmin=-1,vmax=1);axes[0].set_title("Grid goal (+1) and trap (-1)")
rng=np.random.default_rng(SEED);random_returns=[]
for ep in range(200):
 s=start;total=0
 for _ in range(80): s,r,d=step_grid(s,rng.integers(4));total+=r
 if d:break
 random_returns.append(total)
axes[1].hist(random_returns,bins=20,color="#176b66");axes[1].set_title("Random-policy returns");axes[2].bar(["state values","state-action Q values"],[size*size,size*size*4],color=[".6","#d38b45"]);axes[2].set_title("Tabular storage");plt.tight_layout();plt.show()

**Takeaway:** The trap makes shortest-path exploration risky.

**Takeaway:** Random behaviour occasionally succeeds but has poor expected return.

**Takeaway:** Q-learning stores a value for each action in each discrete state.

# 5. Prepare

We create the evaluation boundary before learning any transformation or model choice. Training-only operations remain inside the fitted workflow so held-out rows cannot influence them.

Any learned scaling, encoding, imputation, or resampling belongs after
the split—normally inside a pipeline. Doing it on all rows would let
held-out information influence training and would make evaluation too
optimistic.

In [ ]:
def train_q(epsilon,episodes=2500):
 rng=np.random.default_rng(SEED);Q=np.zeros((size,size,4));returns=[]
 for ep in range(episodes):
  s=start;total=0;eps=epsilon*(1-ep/episodes)+.02
  for _ in range(80):
   a=rng.integers(4) if rng.random()<eps else int(np.argmax(Q[s]));nxt,r,done=step_grid(s,a);target=r+(0 if done else .95*np.max(Q[nxt]));Q[s][a]+=.18*(target-Q[s][a]);s=nxt;total+=r
   if done:break
  returns.append(total)
 return Q,np.array(returns)
print("Fixed seeds: NumPy 0, Gym reset 0, PPO seed 0")

# 6. Train

        Training turns the assumptions behind reinforcement learning: q-learning and ppo into an explicit fitted rule. The printed intermediate values expose what was learned rather than treating fit() as magic.

        Conceptually, fitting follows these steps:

        1. Explore or choose the largest current Q action.
2. Observe reward and next state.
3. Bootstrap toward reward plus discounted next-state value.
4. For PPO, collect trajectories and clip neural-policy probability-ratio updates.

In [ ]:
Q,q_returns=train_q(.8);policy=Q.argmax(2)
class Rewards(BaseCallback):
 def __init__(self): super().__init__();self.values=[]
 def _on_step(self):
  if len(self.model.ep_info_buffer): self.values.append(self.model.ep_info_buffer[-1]["r"])
  return True
callback=Rewards();cart=gym.make("CartPole-v1");ppo=PPO("MlpPolicy",cart,n_steps=128,batch_size=64,n_epochs=4,learning_rate=8e-4,gamma=.98,seed=SEED,verbose=0,device="cpu");ppo.learn(total_timesteps=3500,callback=callback);cart.close();print("Grid final-200 mean:",round(q_returns[-200:].mean(),3),"PPO recorded episodes:",len(callback.values))

# 7. Evaluate

We compare against a deliberately simple baseline, report a task-appropriate metric, and save three diagnostic views that explain where the score came from.

We report a baseline as well as the model and save the central figure as
`key_figure.png`. A model earns attention only by beating a simple,
honestly evaluated alternative.

In [ ]:
def eval_q(Q,n=200):
 vals=[]
 for _ in range(n):
  s=start;total=0
  for _ in range(80): s,r,d=step_grid(s,int(np.argmax(Q[s])));total+=r
  if d:break
  vals.append(total)
 return np.mean(vals)
eval_env=gym.make("CartPole-v1");ppo_returns=[]
for ep in range(20):
 obs,_=eval_env.reset(seed=SEED+ep);total=0;done=False
 while not done:
  action,_=ppo.predict(obs,deterministic=True);obs,r,term,trunc,_=eval_env.step(action);total+=r;done=term or trunc
 ppo_returns.append(total)
eval_env.close();random_cart=22.0;fig,axes=plt.subplots(1,3,figsize=(14,3.7));axes[0].plot(pd.Series(q_returns).rolling(100).mean());axes[0].set_title("Q-learning reward")
arrows=np.array(["↑","↓","←","→"]);axes[1].imshow(np.max(Q,2),cmap="viridis");[(axes[1].text(c,r,arrows[policy[r,c]],ha="center",va="center",color="white")) for r in range(size) for c in range(size) if (r,c) not in [goal,trap]];axes[1].set_title(f"Greedy grid policy · return {eval_q(Q):.2f}")
axes[2].bar(["random CartPole","PPO"],[random_cart,np.mean(ppo_returns)],color=[".6","#176b66"]);axes[2].set_title("20-episode mean return");plt.tight_layout();plt.savefig("key_figure.png",dpi=160,bbox_inches="tight");plt.show();print(f"PPO mean±sd={np.mean(ppo_returns):.1f}±{np.std(ppo_returns):.1f}")

**Takeaway:** The model improves on the simple baseline where its assumptions match the data; the diagnostics show that the aggregate score alone is not enough.

# 8. Break it

We now violate the important condition directly: Without exploration, early arbitrary choices prevent discovery; reward misspecification can teach behaviour you did not intend. A MIP solver often beats RL on operations problems with known constraints. The resulting number makes the cost of that violation visible.

The following experiment deliberately violates an assumption or removes
a protection. Compare the printed damage with the healthy evaluation;
the failure is part of the lesson, not an accidental exception.

In [ ]:
no_explore,bad_returns=train_q(0.0,500);print(f"Exploratory Q policy return={eval_q(Q):.3f}; almost-no-exploration return={eval_q(no_explore):.3f}; states with any learned value={np.any(no_explore!=0,axis=2).sum()}/{size*size}")

**Use this when…** actions change future observations, a simulator or safe interaction exists, and delayed reward truly defines the goal.

        **Don't use this when…** the problem has a reliable optimizer/planner, unsafe exploration, weak reward specification, or only static logged data.

        ## Try this

        1. Move the trap and relearn the grid policy.
2. Set discount to .5 and inspect short-sighted choices.
3. Increase PPO timesteps to 10,000 and plot variance across three seeds.